# USAD Transfer Learning — Sensor 68 Temperatura SIATA (Plan K)

**Objetivo:** Detectar anomalías de temperatura en el sensor SIATA #68 (Jardin Botanico, Medellín)  
usando Transfer Learning desde el modelo USAD preentrenado en SWaT (51 sensores).

**Estrategia Plan K:**
- Submatrix weight transfer (norm-ranked, 612→12 inputs)
- MinMaxScaler [0,1] — misma normalización del modelo original
- Clip físico de temperatura (10–40 °C) para eliminar outliers extremos
- Entrenamiento bifásico: Fase 1 encoder congelado + Fase 2 fine-tuning completo
- Umbral optimizado por **Balanced Accuracy**

In [ ]:
# Celda 1 — Setup Colab: clonar repo y configurar sys.path
import os, sys

REPO_URL = "https://github.com/ronvas234/data-science-monograph.git"
REPO_DIR = "data-science-monograph"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")

if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

USAD_MODULE_PATH = os.path.abspath("modelos/usad")
if USAD_MODULE_PATH not in sys.path:
    sys.path.insert(0, USAD_MODULE_PATH)

print(f"CWD: {os.getcwd()}")
print(f"USAD path en sys.path: {USAD_MODULE_PATH}")
print(f"Archivos USAD: {os.listdir(USAD_MODULE_PATH)}")

In [ ]:
# Celda 2 — Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from dataclasses import dataclass
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    balanced_accuracy_score,
    precision_score, recall_score,
    f1_score, accuracy_score,
    confusion_matrix as sk_confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Celda 3 — Clases del modelo (arquitectura submatrix-compatible)
#
# EncoderNew mantiene las dimensiones internas del pretrained (h1=306, h2=153, z_size=120)
# pero acepta input de 1 sensor: in_size = window_size = 12

class EncoderNew(nn.Module):
    def __init__(self, in_size=12, h1=306, h2=153, z_size=120):
        super().__init__()
        self.linear1 = nn.Linear(in_size, h1)
        self.linear2 = nn.Linear(h1, h2)
        self.linear3 = nn.Linear(h2, z_size)
        self.relu = nn.ReLU(True)

    def forward(self, w):
        out = self.relu(self.linear1(w))
        out = self.relu(self.linear2(out))
        z   = self.relu(self.linear3(out))
        return z


class DecoderNew(nn.Module):
    """Decoder con Sigmoid (compatible con MinMaxScaler [0,1])."""
    def __init__(self, z_size=120, h1=153, h2=306, out_size=12):
        super().__init__()
        self.linear1 = nn.Linear(z_size, h1)
        self.linear2 = nn.Linear(h1, h2)
        self.linear3 = nn.Linear(h2, out_size)
        self.relu    = nn.ReLU(True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, z):
        out = self.relu(self.linear1(z))
        out = self.relu(self.linear2(out))
        w   = self.sigmoid(self.linear3(out))
        return w


class UsadModelNew(nn.Module):
    def __init__(self, w_size=12, z_size=120, h1=306, h2=153):
        super().__init__()
        self.encoder  = EncoderNew(w_size, h1, h2, z_size)
        self.decoder1 = DecoderNew(z_size, h2, h1, w_size)
        self.decoder2 = DecoderNew(z_size, h2, h1, w_size)

    def training_step(self, batch, n):
        z  = self.encoder(batch)
        w1 = self.decoder1(z)
        w2 = self.decoder2(z)
        w3 = self.decoder2(self.encoder(w1))
        loss1 = 1/n * torch.mean((batch-w1)**2) + (1-1/n) * torch.mean((batch-w3)**2)
        loss2 = 1/n * torch.mean((batch-w2)**2) - (1-1/n) * torch.mean((batch-w3)**2)
        return loss1, loss2

    def validation_step(self, batch, n):
        with torch.no_grad():
            z  = self.encoder(batch)
            w1 = self.decoder1(z)
            w2 = self.decoder2(z)
            w3 = self.decoder2(self.encoder(w1))
            loss1 = 1/n * torch.mean((batch-w1)**2) + (1-1/n) * torch.mean((batch-w3)**2)
            loss2 = 1/n * torch.mean((batch-w2)**2) - (1-1/n) * torch.mean((batch-w3)**2)
        return {'val_loss1': loss1.item(), 'val_loss2': loss2.item()}

print("Clases de modelo definidas: EncoderNew, DecoderNew, UsadModelNew")

In [ ]:
# Celda 4 — WeightTransferService: extracción de submatriz norm-ranked

class WeightTransferService:
    """
    Extrae una submatriz del modelo USAD pretrained (51 sensores, w=612)
    para inicializar un modelo de 1 sensor (w=12).
    Estrategia: seleccionar las w_size_new columnas de mayor norma L2
    en la primera capa del encoder (más representativas del espacio latente).
    """

    @staticmethod
    def load_pretrained_state(model_path):
        """Carga el checkpoint y devuelve state_dicts del encoder y decoder1."""
        checkpoint = torch.load(model_path, map_location='cpu')

        # Soporte para distintos formatos de guardado
        if isinstance(checkpoint, dict):
            if 'encoder' in checkpoint and isinstance(checkpoint['encoder'], dict):
                # Formato: {'encoder': state_dict, 'decoder1': state_dict, ...}
                enc_sd = checkpoint['encoder']
                dec_sd = checkpoint['decoder1']
            else:
                # Formato: state_dict completo con prefijos
                enc_sd = {k.replace('encoder.', ''): v
                          for k, v in checkpoint.items() if k.startswith('encoder.')}
                dec_sd = {k.replace('decoder1.', ''): v
                          for k, v in checkpoint.items() if k.startswith('decoder1.')}
        else:
            raise ValueError(f"Formato de checkpoint no reconocido: {type(checkpoint)}")

        return enc_sd, dec_sd

    @staticmethod
    def extract_submatrix(model_path, w_size_new=12):
        """
        Selecciona w_size_new columnas de mayor norma L2 de linear1.weight
        del encoder (shape: [306, 612]) → [306, w_size_new].
        Las mismas filas se extraen de linear3.weight del decoder (shape: [612, 306]).
        """
        enc_sd, dec_sd = WeightTransferService.load_pretrained_state(model_path)

        W1_orig = enc_sd['linear1.weight']  # (306, 612)
        print(f"Encoder linear1 original: {W1_orig.shape}")

        # Norm-ranked: top columnas por norma L2
        col_norms = W1_orig.norm(dim=0)                 # (612,)
        top_cols  = col_norms.topk(w_size_new).indices  # (w_size_new,)
        top_cols_sorted = top_cols.sort().values
        print(f"Top {w_size_new} columnas (índices): {top_cols_sorted.tolist()}")
        print(f"Normas: min={col_norms[top_cols_sorted].min():.4f}  "
              f"max={col_norms[top_cols_sorted].max():.4f}")

        W3_dec_orig = dec_sd['linear3.weight']  # (612, 306)

        new_weights = {
            'encoder': {
                'linear1.weight': W1_orig[:, top_cols_sorted].clone(),   # (306, 12)
                'linear1.bias':   enc_sd['linear1.bias'].clone(),         # (306,)
                'linear2.weight': enc_sd['linear2.weight'].clone(),       # (153, 306)
                'linear2.bias':   enc_sd['linear2.bias'].clone(),         # (153,)
                'linear3.weight': enc_sd['linear3.weight'].clone(),       # (120, 153)
                'linear3.bias':   enc_sd['linear3.bias'].clone(),         # (120,)
            },
            'decoder': {
                'linear1.weight': dec_sd['linear1.weight'].clone(),       # (153, 120)
                'linear1.bias':   dec_sd['linear1.bias'].clone(),         # (153,)
                'linear2.weight': dec_sd['linear2.weight'].clone(),       # (306, 153)
                'linear2.bias':   dec_sd['linear2.bias'].clone(),         # (306,)
                'linear3.weight': W3_dec_orig[top_cols_sorted, :].clone(), # (12, 306)
                'linear3.bias':   dec_sd['linear3.bias'][top_cols_sorted].clone(),  # (12,)
            }
        }
        return new_weights, top_cols_sorted

    @staticmethod
    def load_into_model(model, weights):
        """Carga pesos transferidos en el modelo nuevo."""
        enc = model.encoder
        enc.linear1.weight = nn.Parameter(weights['encoder']['linear1.weight'])
        enc.linear1.bias   = nn.Parameter(weights['encoder']['linear1.bias'])
        enc.linear2.weight = nn.Parameter(weights['encoder']['linear2.weight'])
        enc.linear2.bias   = nn.Parameter(weights['encoder']['linear2.bias'])
        enc.linear3.weight = nn.Parameter(weights['encoder']['linear3.weight'])
        enc.linear3.bias   = nn.Parameter(weights['encoder']['linear3.bias'])
        for dec in [model.decoder1, model.decoder2]:
            dec.linear1.weight = nn.Parameter(weights['decoder']['linear1.weight'].clone())
            dec.linear1.bias   = nn.Parameter(weights['decoder']['linear1.bias'].clone())
            dec.linear2.weight = nn.Parameter(weights['decoder']['linear2.weight'].clone())
            dec.linear2.bias   = nn.Parameter(weights['decoder']['linear2.bias'].clone())
            dec.linear3.weight = nn.Parameter(weights['decoder']['linear3.weight'].clone())
            dec.linear3.bias   = nn.Parameter(weights['decoder']['linear3.bias'].clone())
        return model

print("WeightTransferService definido")

In [ ]:
# Celda 5 — Configuración

@dataclass
class Config:
    # Datos
    csv_path:     str   = 'modelos/usad/data/temperatura_estaciones_2020_2025.csv'
    model_path:   str   = 'modelos/usad/model.pth'
    sensor_id:    int   = 68
    fecha_inicio: str   = '2023-01-01'
    fecha_fin:    str   = '2023-06-30'
    t_clip_min:   float = 10.0    # °C mínimo físicamente posible (Medellín)
    t_clip_max:   float = 40.0    # °C máximo físicamente posible
    # Split: train=ene-abr, val=may, test=jun
    train_end:    str   = '2023-04-30'
    val_start:    str   = '2023-05-01'
    val_end:      str   = '2023-05-31'
    test_start:   str   = '2023-06-01'
    # Arquitectura (mantiene dims internas del pretrained)
    window_size:  int   = 12
    z_size:       int   = 120    # encoder linear3 → (120,153) en model.pth
    h1:           int   = 306
    h2:           int   = 153
    # Entrenamiento
    batch_size:   int   = 128
    epochs_p1:    int   = 50      # Fase 1: encoder congelado
    epochs_p2:    int   = 100     # Fase 2: fine-tuning completo
    lr_p1:        float = 1e-3
    lr2_p1:       float = 3e-4
    lr_p2:        float = 1e-4
    lr2_p2:       float = 3e-5
    lr_patience:  int   = 10
    lr_factor:    float = 0.5
    es_patience:  int   = 20      # Early stopping
    max_grad:     float = 1.0
    alpha:        float = 0.5
    beta:         float = 0.5

cfg = Config()
print(cfg)

In [ ]:
# Celda 6 — Carga y preparación de datos

print("Cargando CSV (puede tardar ~30s en Colab)...")
df_raw = pd.read_csv(
    cfg.csv_path,
    parse_dates=['fecha_hora'],
    dtype={'codigo': 'int32', 't': 'float32', 'calidad': 'int32',
           'calidad_dudosa': 'bool', 'temperatura_dudosa': 'bool'}
)
print(f"CSV cargado: {len(df_raw):,} filas")

# Filtrar sensor y período
df = df_raw[
    (df_raw['codigo'] == cfg.sensor_id) &
    (df_raw['fecha_hora'] >= cfg.fecha_inicio) &
    (df_raw['fecha_hora'] <= cfg.fecha_fin + ' 23:59:59')
].copy().sort_values('fecha_hora').reset_index(drop=True)

print(f"Sensor {cfg.sensor_id} en período: {len(df):,} filas")
print(f"Rango de fechas: {df['fecha_hora'].iloc[0]} → {df['fecha_hora'].iloc[-1]}")
print(f"Temperatura raw: min={df['t'].min():.1f}°C  max={df['t'].max():.1f}°C")
print(f"Anomalías totales: {df['calidad_dudosa'].sum():,} ({df['calidad_dudosa'].mean()*100:.2f}%)")

# Clip físico: elimina valores imposibles (ej. -1000°C por error de sensor)
df['t_clip'] = df['t'].clip(cfg.t_clip_min, cfg.t_clip_max)
n_clipped = (df['t'] != df['t_clip']).sum()
print(f"\nValores clippeados ({cfg.t_clip_min}–{cfg.t_clip_max}°C): {n_clipped:,}")

# Split cronológico por mes (sin data leakage)
df_train = df[df['fecha_hora'] <= cfg.train_end + ' 23:59:59'].reset_index(drop=True)
df_val   = df[(df['fecha_hora'] >= cfg.val_start) &
              (df['fecha_hora'] <= cfg.val_end + ' 23:59:59')].reset_index(drop=True)
df_test  = df[df['fecha_hora'] >= cfg.test_start].reset_index(drop=True)

total = len(df_train) + len(df_val) + len(df_test)
print(f"\nSplit:")
print(f"  Train: {len(df_train):>8,} ({len(df_train)/total*100:.1f}%)  "
      f"anomalías: {df_train['calidad_dudosa'].sum():,} ({df_train['calidad_dudosa'].mean()*100:.1f}%)")
print(f"  Val:   {len(df_val):>8,} ({len(df_val)/total*100:.1f}%)  "
      f"anomalías: {df_val['calidad_dudosa'].sum():,} ({df_val['calidad_dudosa'].mean()*100:.1f}%)")
print(f"  Test:  {len(df_test):>8,} ({len(df_test)/total*100:.1f}%)  "
      f"anomalías: {df_test['calidad_dudosa'].sum():,} ({df_test['calidad_dudosa'].mean()*100:.1f}%)")

# MinMaxScaler ajustado SOLO sobre datos normales de train
scaler = MinMaxScaler()
normal_train_vals = df_train.loc[~df_train['calidad_dudosa'], 't_clip'].values.reshape(-1, 1)
scaler.fit(normal_train_vals)
print(f"\nMinMaxScaler ajustado sobre {len(normal_train_vals):,} muestras normales de train")
print(f"  Rango: [{scaler.data_min_[0]:.2f}°C, {scaler.data_max_[0]:.2f}°C] → [0, 1]")

# Normalizar todos los conjuntos
for df_split in [df_train, df_val, df_test]:
    df_split['t_norm'] = scaler.transform(
        df_split['t_clip'].values.reshape(-1, 1)
    ).flatten().astype(np.float32)

In [ ]:
# Celda 7 — Creación de ventanas deslizantes y DataLoaders

def make_windows(series, flags, window_size):
    """Ventanas de stride=1. Label=1 si CUALQUIER timestep es anomalía."""
    n   = len(series) - window_size + 1
    idx = np.arange(window_size)[None, :] + np.arange(n)[:, None]  # (n, W)
    windows = series[idx].astype(np.float32)    # (n, W)
    labels  = flags[idx].any(axis=1).astype(np.int8)  # (n,)
    return windows, labels

W_train, L_train = make_windows(
    df_train['t_norm'].values, df_train['calidad_dudosa'].values, cfg.window_size)
W_val,   L_val   = make_windows(
    df_val['t_norm'].values,   df_val['calidad_dudosa'].values,   cfg.window_size)
W_test,  L_test  = make_windows(
    df_test['t_norm'].values,  df_test['calidad_dudosa'].values,  cfg.window_size)

# Train: solo ventanas normales (sin anomalías)
normal_mask     = L_train == 0
W_train_normal  = W_train[normal_mask]

train_ds = TensorDataset(torch.from_numpy(W_train_normal))
val_ds   = TensorDataset(torch.from_numpy(W_val))
test_ds  = TensorDataset(torch.from_numpy(W_test))

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False)

print(f"Ventanas (stride=1):")
print(f"  Train normal: {len(W_train_normal):,}")
print(f"  Val  (todas): {len(W_val):,}  |  anomalías: {L_val.sum():,} ({L_val.mean()*100:.1f}%)")
print(f"  Test (todas): {len(W_test):,}  |  anomalías: {L_test.sum():,} ({L_test.mean()*100:.1f}%)")

In [ ]:
# Celda 8 — Inicializar modelo con pesos transferidos (submatrix TL)

pretrained_weights, top_cols = WeightTransferService.extract_submatrix(
    cfg.model_path, cfg.window_size
)

model = UsadModelNew(cfg.window_size, cfg.z_size, cfg.h1, cfg.h2).to(device)
model = WeightTransferService.load_into_model(model, pretrained_weights)

total_params     = sum(p.numel() for p in model.parameters())
encoder_params   = sum(p.numel() for p in model.encoder.parameters())
decoder_params   = sum(p.numel() for p in model.decoder1.parameters())

print(f"\nModelo inicializado con pesos transferidos:")
print(f"  Encoder linear1: {model.encoder.linear1.weight.shape}  (submatriz de 612→{cfg.window_size})")
print(f"  Decoder linear3: {model.decoder1.linear3.weight.shape}  (submatriz de 612→{cfg.window_size})")
print(f"  Parámetros encoder:   {encoder_params:,}")
print(f"  Parámetros decoder:   {decoder_params:,}")
print(f"  Parámetros totales:   {total_params:,}")

In [ ]:
# Celda 9 — Entrenamiento bifásico

def run_training(model, train_loader, val_loader, cfg, device):
    history = {'val_loss1': [], 'val_loss2': []}
    phase1_end_epoch = cfg.epochs_p1

    # ---- FASE 1: encoder congelado ----
    print("=" * 55)
    print(f"FASE 1: encoder congelado ({cfg.epochs_p1} épocas)")
    print("=" * 55)
    for p in model.encoder.parameters():
        p.requires_grad = False

    opt1 = torch.optim.Adam(model.decoder1.parameters(), lr=cfg.lr_p1)
    opt2 = torch.optim.Adam(model.decoder2.parameters(), lr=cfg.lr2_p1)
    sch1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt1, factor=cfg.lr_factor, patience=cfg.lr_patience, verbose=False)
    sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt2, factor=cfg.lr_factor, patience=cfg.lr_patience, verbose=False)

    for epoch in range(1, cfg.epochs_p1 + 1):
        model.train()
        for [batch] in train_loader:
            batch = batch.to(device)
            loss1, _ = model.training_step(batch, epoch)
            opt1.zero_grad()
            loss1.backward()
            nn.utils.clip_grad_norm_(model.decoder1.parameters(), cfg.max_grad)
            opt1.step()

            _, loss2 = model.training_step(batch, epoch)
            opt2.zero_grad()
            loss2.backward()
            nn.utils.clip_grad_norm_(model.decoder2.parameters(), cfg.max_grad)
            opt2.step()

        model.eval()
        val_outs = [model.validation_step(b.to(device), epoch) for [b] in val_loader]
        vl1 = float(np.mean([x['val_loss1'] for x in val_outs]))
        vl2 = float(np.mean([x['val_loss2'] for x in val_outs]))
        sch1.step(vl1); sch2.step(vl2)
        history['val_loss1'].append(vl1)
        history['val_loss2'].append(vl2)
        if epoch % 10 == 0:
            print(f"  Época {epoch:3d}/{cfg.epochs_p1} | "
                  f"val_loss1={vl1:.6f}  val_loss2={vl2:.6f}")

    # ---- FASE 2: fine-tuning completo ----
    print("\n" + "=" * 55)
    print(f"FASE 2: fine-tuning completo (hasta {cfg.epochs_p2} épocas, ES={cfg.es_patience})")
    print("=" * 55)
    for p in model.encoder.parameters():
        p.requires_grad = True

    params1 = list(model.encoder.parameters()) + list(model.decoder1.parameters())
    params2 = list(model.encoder.parameters()) + list(model.decoder2.parameters())
    opt1 = torch.optim.Adam(params1, lr=cfg.lr_p2)
    opt2 = torch.optim.Adam(params2, lr=cfg.lr2_p2)
    sch1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt1, factor=cfg.lr_factor, patience=cfg.lr_patience, verbose=False)
    sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt2, factor=cfg.lr_factor, patience=cfg.lr_patience, verbose=False)

    best_vl1 = float('inf')
    no_improve = 0
    best_state = None

    for local_ep in range(1, cfg.epochs_p2 + 1):
        global_ep = cfg.epochs_p1 + local_ep
        model.train()
        for [batch] in train_loader:
            batch = batch.to(device)
            loss1, _ = model.training_step(batch, local_ep)
            opt1.zero_grad()
            loss1.backward()
            nn.utils.clip_grad_norm_(params1, cfg.max_grad)
            opt1.step()

            _, loss2 = model.training_step(batch, local_ep)
            opt2.zero_grad()
            loss2.backward()
            nn.utils.clip_grad_norm_(params2, cfg.max_grad)
            opt2.step()

        model.eval()
        val_outs = [model.validation_step(b.to(device), local_ep) for [b] in val_loader]
        vl1 = float(np.mean([x['val_loss1'] for x in val_outs]))
        vl2 = float(np.mean([x['val_loss2'] for x in val_outs]))
        sch1.step(vl1); sch2.step(vl2)
        history['val_loss1'].append(vl1)
        history['val_loss2'].append(vl2)

        if vl1 < best_vl1:
            best_vl1 = vl1
            no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1

        if local_ep % 10 == 0:
            print(f"  Época {global_ep:3d} (F2 {local_ep:3d}) | "
                  f"val_loss1={vl1:.6f}  val_loss2={vl2:.6f}  "
                  f"ES={no_improve}/{cfg.es_patience}")

        if no_improve >= cfg.es_patience:
            print(f"\n  Early stopping en época global {global_ep} "
                  f"(mejor val_loss1={best_vl1:.6f})")
            break

    # Restaurar mejor estado
    if best_state is not None:
        model.load_state_dict(best_state)
        print("  Modelo restaurado al mejor estado.")

    history['phase1_end'] = phase1_end_epoch
    return history


history = run_training(model, train_loader, val_loader, cfg, device)
print(f"\nEntrenamiento completado. Total épocas: {len(history['val_loss1'])}")

In [ ]:
# Celda 10 — Gráfica 1: Serie temporal con splits y anomalías reales

fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(df_train['fecha_hora'], df_train['t_clip'],
        color='steelblue', lw=0.5, label='Train')
ax.plot(df_val['fecha_hora'],   df_val['t_clip'],
        color='darkorange', lw=0.5, label='Val')
ax.plot(df_test['fecha_hora'],  df_test['t_clip'],
        color='crimson', lw=0.5, label='Test')

anom_all = df[df['calidad_dudosa']]
ax.scatter(anom_all['fecha_hora'], anom_all['t_clip'],
           color='purple', s=5, zorder=5, alpha=0.7, label='Anomalía real')

n_tr, n_va, n_te = len(df_train), len(df_val), len(df_test)
ax.set_title(
    f'Sensor {cfg.sensor_id} — Temperatura (°C)\n'
    f'Train: {n_tr:,} | Val: {n_va:,} | Test: {n_te:,}',
    fontsize=13
)
ax.set_xlabel('Fecha')
ax.set_ylabel('Temperatura (°C)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grafica_01_splits.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Celda 11 — Gráfica 2: Curvas de entrenamiento (Fase 1 / Fase 2)

epochs_all = list(range(1, len(history['val_loss1']) + 1))
p1_end     = history['phase1_end']

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(epochs_all, history['val_loss1'], '-x', color='steelblue',
        markersize=3, label='val_loss1 (AE1)')
ax.plot(epochs_all, history['val_loss2'], '-x', color='darkorange',
        markersize=3, label='val_loss2 (AE2)')
ax.axvline(p1_end, color='gray', linestyle='--', lw=1.5, label='Fase 1 / Fase 2')
ax.set_xlabel('Época')
ax.set_ylabel('Loss')
ax.set_title(
    f'Curva de entrenamiento — USAD Transfer Learning Sensor {cfg.sensor_id}',
    fontsize=13
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grafica_02_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Celda 12 — Inferencia: cálculo de scores de anomalía

def compute_scores(model, loader, device, alpha=0.5, beta=0.5):
    model.eval()
    scores, recons = [], []
    with torch.no_grad():
        for [batch] in loader:
            batch = batch.to(device)
            w1 = model.decoder1(model.encoder(batch))
            w2 = model.decoder2(model.encoder(w1))
            s  = (alpha * torch.mean((batch - w1)**2, dim=1) +
                  beta  * torch.mean((batch - w2)**2, dim=1))
            scores.extend(s.cpu().numpy())
            # Último timestep de la reconstrucción para visualización
            recons.extend(w1[:, -1].cpu().numpy())
    return np.array(scores), np.array(recons)

scores_val,  recon_val_norm  = compute_scores(model, val_loader,  device, cfg.alpha, cfg.beta)
scores_test, recon_test_norm = compute_scores(model, test_loader, device, cfg.alpha, cfg.beta)

# Denormalizar reconstrucción
recon_val_c  = scaler.inverse_transform(recon_val_norm.reshape(-1, 1)).flatten()
recon_test_c = scaler.inverse_transform(recon_test_norm.reshape(-1, 1)).flatten()

# Temperatura original (último timestep de cada ventana)
orig_val_c  = df_val['t_clip'].values[cfg.window_size - 1:]
orig_test_c = df_test['t_clip'].values[cfg.window_size - 1:]

# Timestamps de cada ventana (último timestep)
dates_val  = df_val['fecha_hora'].values[cfg.window_size - 1:]
dates_test = df_test['fecha_hora'].values[cfg.window_size - 1:]

df_val_w  = pd.DataFrame({'fecha': dates_val,  'error': scores_val,
                           'flag': L_val, 't_orig': orig_val_c, 't_recon': recon_val_c})
df_test_w = pd.DataFrame({'fecha': dates_test, 'error': scores_test,
                           'flag': L_test, 't_orig': orig_test_c, 't_recon': recon_test_c})

print(f"Scores val  | normal: {scores_val[L_val==0].mean():.6f}  "
      f"anomalía: {scores_val[L_val==1].mean():.6f}")
print(f"Scores test | normal: {scores_test[L_test==0].mean():.6f}  "
      f"anomalía: {scores_test[L_test==1].mean():.6f}")
print(f"\n¿Anomalías > Normal? Val: {scores_val[L_val==1].mean() > scores_val[L_val==0].mean()}  "
      f"Test: {scores_test[L_test==1].mean() > scores_test[L_test==0].mean()}")

In [ ]:
# Celda 13 — Selección de umbral por Balanced Accuracy (sobre validación)

# Candidatos: percentiles 50 → 99.9 del error en validación
pct_values   = np.linspace(50, 99.9, 500)
thresholds   = np.percentile(scores_val, pct_values)
ba_scores    = np.array([
    balanced_accuracy_score(L_val, (scores_val >= t).astype(int))
    for t in thresholds
])

best_idx = int(np.argmax(ba_scores))
umbral   = float(thresholds[best_idx])
best_ba  = float(ba_scores[best_idx])

print(f"Umbral óptimo (Balanced Accuracy en Val): θ = {umbral:.6f}")
print(f"Balanced Accuracy en validación con θ:       {best_ba:.4f}")
print(f"Percentil correspondiente: p{pct_values[best_idx]:.1f}")

In [ ]:
# Celda 14 — Gráfica 3: Curva ROC + umbral Youden's J

fpr, tpr, thr_roc = roc_curve(L_val, scores_val)
auc_val = roc_auc_score(L_val, scores_val)

# Umbral Youden's J: maximiza TPR - FPR
j_scores = tpr - fpr
j_idx    = int(np.argmax(j_scores))
thr_youden = float(thr_roc[j_idx]) if j_idx < len(thr_roc) else float(thr_roc[-1])

print(f"Umbral óptimo (Youden's J): {thr_youden:.6f} | TPR={tpr[j_idx]:.4f}  FPR={fpr[j_idx]:.4f}  AUC={auc_val:.4f}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC Curve (AUC={auc_val:.4f})')
ax.scatter(fpr[j_idx], tpr[j_idx], color='red', s=80, zorder=5,
           label=f"Youden's J (thr={thr_youden:.4f})")
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatorio')
ax.set_xlabel('FPR (1 − Especificidad)')
ax.set_ylabel('TPR (Sensibilidad)')
ax.set_title(f'ROC Curve — USAD TL Sensor {cfg.sensor_id} Plan K', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grafica_03_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Celda 15 — Gráficas 4 y 5: Error de reconstrucción (Validación y Test)

def plot_error_reconstruction(df_w, threshold, label, show_pred=False, save_path=None):
    fig, ax = plt.subplots(figsize=(14, 4))

    dates = pd.to_datetime(df_w['fecha'])
    ax.plot(dates, df_w['error'], color='steelblue', lw=0.6,
            label='Error de reconstrucción')
    ax.axhline(threshold, color='black', linestyle='--', lw=1.5,
               label=f'Umbral θ = {threshold:.4f}')

    # Anomalías reales
    anom_real = df_w[df_w['flag'] == 1]
    ax.scatter(pd.to_datetime(anom_real['fecha']), anom_real['error'],
               color='red', s=12, zorder=5, label='Anomalías reales')

    if show_pred:
        anom_pred = df_w[df_w['error'] >= threshold]
        ax.scatter(pd.to_datetime(anom_pred['fecha']), anom_pred['error'],
                   color='purple', marker='x', s=20, zorder=6,
                   label='Anomalías predichas')

    ax.set_title(f'Errores de reconstrucción — conjunto {label}', fontsize=13)
    ax.set_xlabel('Tiempo')
    ax.set_ylabel('Error')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%Y'))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# P2: umbral Youden's J sobre validación (referencia visual)
plot_error_reconstruction(
    df_val_w, umbral, 'Validación', show_pred=False,
    save_path='grafica_04_error_val.png'
)
plot_error_reconstruction(
    df_test_w, umbral, 'Test', show_pred=True,
    save_path='grafica_05_error_test.png'
)

In [ ]:
# Celda 16 — Gráfica 6: Reconstrucción vs Original (período completo)

fig, ax = plt.subplots(figsize=(16, 4))

# Señal original completa
ax.plot(df['fecha_hora'], df['t_clip'],
        color='steelblue', lw=0.4, alpha=0.8, label='Original')

# Reconstrucción en val y test
ax.plot(pd.to_datetime(df_val_w['fecha']),  df_val_w['t_recon'],
        color='orange', lw=0.6, alpha=0.9, label='Reconstruida')
ax.plot(pd.to_datetime(df_test_w['fecha']), df_test_w['t_recon'],
        color='orange', lw=0.6, alpha=0.9)

# Anomalías reales
anom_all = df[df['calidad_dudosa']]
ax.scatter(anom_all['fecha_hora'], anom_all['t_clip'],
           color='red', s=6, zorder=5, label='Anomalías reales')

ax.set_title(
    f'Reconstrucción vs Original + Anomalías — USAD TL Sensor {cfg.sensor_id}',
    fontsize=13
)
ax.set_xlabel('Fecha')
ax.set_ylabel('Temperatura (°C)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grafica_06_recon_completa.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Celda 17 — Gráfica 7: Reconstrucción vs Original — zoom Validación

fig, ax = plt.subplots(figsize=(14, 4))

dates_v = pd.to_datetime(df_val_w['fecha'])
ax.plot(dates_v, df_val_w['t_orig'],  color='steelblue', lw=1.0, label='Original')
ax.plot(dates_v, df_val_w['t_recon'], color='orange',    lw=1.0, label='Reconstruida')

anom_val_plot = df_val_w[df_val_w['flag'] == 1]
ax.scatter(pd.to_datetime(anom_val_plot['fecha']), anom_val_plot['t_orig'],
           color='red', s=18, zorder=5, label='Anomalías reales')

ax.set_title(
    f'Reconstrucción vs Original + Anomalías — USAD TL (Validación)',
    fontsize=13
)
ax.set_xlabel('Fecha')
ax.set_ylabel('Temperatura (°C)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%Y'))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grafica_07_recon_val.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Celda 18 — Gráfica 8: Reconstrucción vs Original — zoom Test + predicciones

df_test_w['flag_pred'] = (df_test_w['error'] >= umbral).astype(int)

fig, ax = plt.subplots(figsize=(14, 4))

dates_t = pd.to_datetime(df_test_w['fecha'])
ax.plot(dates_t, df_test_w['t_orig'],  color='steelblue', lw=1.0, label='Original')
ax.plot(dates_t, df_test_w['t_recon'], color='orange',    lw=1.0, label='Reconstruida')

anom_real_t = df_test_w[df_test_w['flag'] == 1]
anom_pred_t = df_test_w[df_test_w['flag_pred'] == 1]

ax.scatter(pd.to_datetime(anom_real_t['fecha']), anom_real_t['t_orig'],
           color='red', s=18, zorder=5, label='Anomalías reales')
ax.scatter(pd.to_datetime(anom_pred_t['fecha']), anom_pred_t['t_recon'],
           color='purple', marker='x', s=30, zorder=6, label='Anomalías predichas')

ax.set_title(
    f'Reconstrucción vs Original + Anomalías — USAD TL (Test)',
    fontsize=13
)
ax.set_xlabel('Fecha')
ax.set_ylabel('Temperatura (°C)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%Y'))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('grafica_08_recon_test.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Celda 19 — Gráfica 9: Métricas de clasificación (Val y Test)

def compute_and_print_metrics(y_true, y_scores, threshold, label):
    y_pred = (y_scores >= threshold).astype(int)
    cm     = sk_confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    print(f"=== Métricas — Conjunto de {label} ===")
    print(f"  Accuracy:      {acc:.4f}")
    print(f"  Balanced Acc:  {bacc:.4f}")
    print(f"  Precision:     {prec:.4f}")
    print(f"  Recall:        {rec:.4f}")
    print(f"  F1:            {f1:.4f}")
    print()
    print(f"  Confusion Matrix:")
    print(f"  {'':20s} Real Normal   Real Anomalía")
    print(f"  {'Pred Normal':20s} {tn:>12,}  {fn:>13,}")
    print(f"  {'Pred Anomalía':20s} {fp:>12,}  {tp:>13,}")
    print()
    return {'acc': acc, 'bacc': bacc, 'prec': prec, 'rec': rec,
            'f1': f1, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}


df_val_w['flag_pred']  = (df_val_w['error']  >= umbral).astype(int)

metrics_val  = compute_and_print_metrics(L_val,   scores_val,  umbral, 'Validación')
metrics_test = compute_and_print_metrics(L_test,  scores_test, umbral, 'Test')

# Tabla resumen
summary = pd.DataFrame(
    [metrics_val, metrics_test],
    index=['Validación', 'Test']
)[['acc', 'bacc', 'prec', 'rec', 'f1', 'TP', 'TN', 'FP', 'FN']]
summary.columns = ['Accuracy', 'Balanced Acc', 'Precision', 'Recall', 'F1',
                   'TP', 'TN', 'FP', 'FN']
print("=== Resumen ===")
print(summary.to_string(float_format='{:.4f}'.format))

In [ ]:
# Celda 20 — Guardar modelo fine-tuneado

torch.save(
    {
        'model_state_dict': model.state_dict(),
        'scaler_min':  scaler.data_min_[0],
        'scaler_max':  scaler.data_max_[0],
        'threshold':   umbral,
        'top_cols':    top_cols.tolist(),
        'config':      cfg.__dict__,
        'metrics_val': metrics_val,
        'metrics_test': metrics_test,
    },
    'modelos/usad/model_tl_sensor68_planK.pth'
)
print("Modelo guardado: modelos/usad/model_tl_sensor68_planK.pth")
print(f"Umbral: {umbral:.6f}")
print(f"Balanced Accuracy (Val):  {metrics_val['bacc']:.4f}")
print(f"Balanced Accuracy (Test): {metrics_test['bacc']:.4f}")